In [1]:
"""
fano_trb_algebra_v2.py
========================
Corrected analysis: enumerate all elements over Z/3Z, classify
zero divisors, and identify which candidate is uniquely correct.

Key criteria:
  1. Does t annihilate the Fano sector? (bad)
  2. Is t nilpotent with minimal degree? (good)
  3. Is t chiral? (required)
  4. Is t a genuine zero divisor, not just 3*v? (good)
  5. Is the Fano subalgebra closed? (required)
"""

import numpy as np
from itertools import product

MOD9 = 9
MOD3 = 3


def build_mult():
    mult = np.zeros((9, 9, 9), dtype=int)
    for i in range(9):
        mult[7][i][i] = 1
        mult[i][7][i] = 1
    lines = [
        (0, 1, 2), (0, 3, 4), (0, 5, 6),
        (1, 3, 5), (1, 4, 6), (2, 3, 6), (2, 4, 5)
    ]
    for (a, b, c) in lines:
        mult[a][b][c] = 1
        mult[b][c][a] = 1
        mult[c][a][b] = 1
        mult[b][a][c] = -1 % MOD9
        mult[c][b][a] = -1 % MOD9
        mult[a][c][b] = -1 % MOD9
    for i in range(7):
        mult[i][i][7] = -1 % MOD9
    return mult


def set_t(mult, t_sq, t_left, t_right):
    mult[8, 8, :] = 0
    mult[8, 8, 7] = t_sq % MOD9
    for i in range(7):
        mult[8, i, :] = 0
        mult[i, 8, :] = 0
        mult[8][i][i] = t_left % MOD9
        mult[i][8][i] = t_right % MOD9


def mat_left(coeffs, mult, mod):
    A = np.zeros((9, 9), dtype=int)
    for k in range(9):
        for i in range(9):
            s = 0
            for j in range(9):
                s += coeffs[j] * mult[j][i][k]
            A[k, i] = s % mod
    return A


def mat_right(coeffs, mult, mod):
    A = np.zeros((9, 9), dtype=int)
    for k in range(9):
        for i in range(9):
            s = 0
            for j in range(9):
                s += coeffs[j] * mult[i][j][k]
            A[k, i] = s % mod
    return A


def rank_mod_p(A, p):
    B = A.copy() % p
    n = B.shape[0]
    rank = 0
    for col in range(n):
        pivot = None
        for r in range(rank, n):
            if B[r, col] % p != 0:
                pivot = r
                break
        if pivot is None:
            continue
        B[[rank, pivot]] = B[[pivot, rank]]
        inv = pow(int(B[rank, col]), -1, p)
        B[rank] = (B[rank] * inv) % p
        for r in range(n):
            if r != rank and B[r, col] % p != 0:
                f = B[r, col]
                B[r] = (B[r] - f * B[rank]) % p
        rank += 1
    return rank


def mul_coeffs(x, y, mult, mod):
    res = np.zeros(9, dtype=int)
    for i in range(9):
        for j in range(9):
            for k in range(9):
                res[k] += x[i] * y[j] * mult[i][j][k]
    return res % mod


def analyze(name, t_sq, t_left, t_right):
    print(f"\n=== {name} ===")
    mult = build_mult()
    set_t(mult, t_sq, t_left, t_right)
    mult3 = mult % 3

    n_nonzero = 0
    n_invertible = 0
    n_left_only = 0
    n_right_only = 0
    n_both_zd = 0

    for coeffs in product(range(3), repeat=9):
        if all(c == 0 for c in coeffs):
            continue
        n_nonzero += 1
        AL = mat_left(coeffs, mult3, MOD3)
        AR = mat_right(coeffs, mult3, MOD3)
        rL = rank_mod_p(AL, 3)
        rR = rank_mod_p(AR, 3)
        if rL == 9 and rR == 9:
            n_invertible += 1
        elif rL < 9 and rR < 9:
            n_both_zd += 1
        elif rL < 9:
            n_left_only += 1
        else:
            n_right_only += 1

    print(f"  Non-zero:                  {n_nonzero}")
    print(f"  Invertible:                {n_invertible}")
    print(f"  Left-only zero divisors:   {n_left_only}")
    print(f"  Right-only zero divisors:  {n_right_only}")
    print(f"  Both-sides zero divisors:  {n_both_zd}")

    # Does t annihilate Fano?
    t_annihilates_fano = all(
        mult[8][i][k] == 0 and mult[i][8][k] == 0
        for i in range(7) for k in range(7)
    )
    print(f"  t annihilates Fano:        {t_annihilates_fano}")

    # Chiral?
    chiral = any(
        mult[8][i][k] != mult[i][8][k]
        for i in range(7) for k in range(9)
    )
    print(f"  Chiral t-action:           {chiral}")

    # Nilpotency of t over Z9
    t = np.zeros(9, dtype=int); t[8] = 1
    cur = t.copy()
    nilpotency = None
    for p in range(2, 6):
        cur = mul_coeffs(cur, t, mult, MOD9)
        if all(c == 0 for c in cur):
            nilpotency = p
            break
    print(f"  Nilpotency degree of t:    {nilpotency if nilpotency else '>5'}")

    # Count "genuine" zero divisors (not of form 3*v)
    # These are elements whose t-coefficient is in {1, 2} (mod 3 units)
    # and which have rank < 9 mod 3
    genuine_zd = 0
    for coeffs in product(range(3), repeat=9):
        if coeffs[8] == 0:
            continue  # skip the 3-ideal
        if all(c == 0 for c in coeffs):
            continue
        AL = mat_left(coeffs, mult3, MOD3)
        rL = rank_mod_p(AL, 3)
        if rL < 9:
            genuine_zd += 1
    print(f"  Genuine zero divisors:     {genuine_zd}")


candidates = [
    ("A: t²=0, t·e=0, e·t=0",           0, 0, 0),
    ("B: t²=-1, t·e=0, e·t=0",          8, 0, 0),
    ("C: t²=3, t·e=3e, e·t=3e",         3, 3, 3),
    ("D: t²=6, t·e=6e, e·t=6e",         6, 6, 6),
    ("E: t²=0, t·e=3e, e·t=-3e",        0, 3, 6),
    ("F: t²=-1, t·e=e, e·t=e",          8, 1, 1),
    ("G: t²=-1, t·e=e, e·t=-e",         8, 1, 8),
]

print("=" * 60)
print("FANO-TRB ALGEBRA OVER Z/9Z — CORRECTED ANALYSIS")
print("=" * 60)

for name, t_sq, t_left, t_right in candidates:
    analyze(name, t_sq, t_left, t_right)

print("\n" + "=" * 60)
print("DECISION CRITERIA")
print("=" * 60)
print("""
Correct candidate must have:
  ✓ t does NOT annihilate Fano (Fano must survive)
  ✓ t is chiral (t·e ≠ e·t)
  ✓ t is nilpotent (finite degree, drains cleanly)
  ✓ t is a genuine zero divisor (not just 3·v)
  ✓ Fano subalgebra closed

Run the script and check which candidate satisfies all five.
""")

FANO-TRB ALGEBRA OVER Z/9Z — CORRECTED ANALYSIS

=== A: t²=0, t·e=0, e·t=0 ===
  Non-zero:                  19682
  Invertible:                6858
  Left-only zero divisors:   0
  Right-only zero divisors:  0
  Both-sides zero divisors:  12824
  t annihilates Fano:        True
  Chiral t-action:           False
  Nilpotency degree of t:    2
  Genuine zero divisors:     8550

=== B: t²=-1, t·e=0, e·t=0 ===
  Non-zero:                  19682
  Invertible:                5202
  Left-only zero divisors:   0
  Right-only zero divisors:  0
  Both-sides zero divisors:  14480
  t annihilates Fano:        True
  Chiral t-action:           False
  Nilpotency degree of t:    >5
  Genuine zero divisors:     10206

=== C: t²=3, t·e=3e, e·t=3e ===
  Non-zero:                  19682
  Invertible:                6858
  Left-only zero divisors:   0
  Right-only zero divisors:  0
  Both-sides zero divisors:  12824
  t annihilates Fano:        False
  Chiral t-action:           False
  Nilpotency degre